In [1]:
# importing pandas
import pandas as pd

In [2]:
# loading data
messages = pd.read_csv("./smsspamcollection/SMSSpamCollection", sep= '\t', names= ["label", "message"])

In [3]:
# importing nlp libraries (1)
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nipuntewari/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# importing nlp libraries (2)
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [5]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# instantiating the imports 
ps = PorterStemmer()

In [13]:
corpus = [] # final corpus that we will use
for i in range(0, len(messages)):
    review = re.sub('^[a-zA-z]', ' ', messages['message'][i]) # removing unnecesary characters
    review = review.lower() # lowering the text for cleanup
    review = review.split() # creating a list
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')] # removing all stopwords
    review = ' '.join(review) # converting it back to a sentence
    corpus.append(review) # appending the message to the corpus

In [15]:
# Creating Bag of words model
from sklearn.feature_extraction.text import CountVectorizer

In [19]:
# instantiating and limiting to most frequent 2500 features
cv = CountVectorizer(max_features= 2500)

In [20]:
# creating our feature set and converting it to an array
X = cv.fit_transform(corpus).toarray()

In [21]:
# converting the labels to dummies
y = pd.get_dummies(messages['label'])

In [22]:
y.head()

,ham,spam
0,1,0
1,1,0
2,0,1
3,1,0
4,1,0


In [23]:
# taking just the spam column
y = y.iloc[:, 1].values

In [24]:
# train test split
from sklearn.model_selection import train_test_split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 23)

In [27]:
# train model using naive bayes
from sklearn.naive_bayes import MultinomialNB

In [28]:
spam_model = MultinomialNB()
spam_model.fit(X_train, y_train)

MultinomialNB()

In [29]:
y_pred = spam_model.predict(X_test)

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

In [31]:
confusion_matrix(y_test, y_pred)

array([[949,  10],
       [  9, 147]])

In [33]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       959
           1       0.94      0.94      0.94       156

    accuracy                           0.98      1115
   macro avg       0.96      0.97      0.96      1115
weighted avg       0.98      0.98      0.98      1115

